# LFR Evaluation Results

Comparison of 3 methods across data regimes with mean ± std (4 seeds).

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

ROOT         = project_root
RESULTS_DIR  = ROOT / "results"
REGIME_ORDER  = ["1", "10", "20", "100", "all"]
REGIME_LABELS = ["1 SPC", "10 SPC", "20 SPC", "100 SPC", "All data"]

# ── Load & merge results ───────────────────────────────────────────────────
lfr_df  = pd.read_csv(RESULTS_DIR / "lfr_eval_results.csv")
sup_df  = pd.read_csv(RESULTS_DIR / "supervised_results.csv")
df      = pd.concat([lfr_df, sup_df], ignore_index=True)
df["regime"] = pd.Categorical(df["regime"].astype(str), categories=REGIME_ORDER, ordered=True)
df = df.sort_values(["method", "regime"]).reset_index(drop=True)

METHODS = {
    "lfr_linear_eval":   dict(color="#4C72B0", marker="o", label="LFR – Linear Eval"),
    "lfr_full_finetune": dict(color="#DD8452", marker="s", label="LFR – Full Finetune"),
    "supervised":        dict(color="#55A868", marker="^", label="Supervised (scratch)"),
}

# Compute mean ± std per (method, regime) for each metric
stats = (
    df.groupby(["method", "regime"])[["test_acc", "test_f1", "test_recall", "test_precision"]]
    .agg(["mean", "std"])
    .reset_index()
)
stats.columns = ["method", "regime"] + [f"{m}_{s}" for m, s in stats.columns[2:]]
stats["regime"] = pd.Categorical(stats["regime"], categories=REGIME_ORDER, ordered=True)
stats = stats.sort_values(["method", "regime"]).reset_index(drop=True)

print("Loaded methods:", df["method"].unique().tolist())
print(stats[["method","regime","test_acc_mean","test_acc_std"]].to_string(index=False))


In [ ]:
# ── Line plots: mean ± std for all metrics ────────────────────────────────
METRICS = [
    ("test_acc",       "Accuracy"),
    ("test_f1",        "F1 (macro)"),
    ("test_recall",    "Recall (macro)"),
    ("test_precision", "Precision (macro)"),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Downstream Evaluation — MotionSense (mean ± std, 4 seeds)", fontsize=14, fontweight="bold")

for ax, (metric, title) in zip(axes.flat, METRICS):
    for method, style in METHODS.items():
        sub = stats[stats["method"] == method].set_index("regime").loc[REGIME_ORDER]
        mu  = sub[f"{metric}_mean"].values
        sd  = sub[f"{metric}_std"].fillna(0).values
        ax.plot(REGIME_LABELS, mu, color=style["color"], marker=style["marker"],
                label=style["label"], linewidth=2)
        ax.fill_between(range(len(REGIME_LABELS)), mu - sd, mu + sd,
                        color=style["color"], alpha=0.15)
    ax.set_title(title); ax.set_ylabel(title); ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eval_lines_all_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eval_lines_all_metrics.png")


In [ ]:
# ── Bar chart: accuracy comparison (3 methods × regimes) ──────────────────
x = np.arange(len(REGIME_LABELS))
w = 0.25
fig, ax = plt.subplots(figsize=(13, 5))
fig.suptitle("Test Accuracy — MotionSense (mean ± std, 4 seeds)", fontsize=13, fontweight="bold")

for i, (method, style) in enumerate(METHODS.items()):
    sub = stats[stats["method"] == method].set_index("regime").loc[REGIME_ORDER]
    mu  = sub["test_acc_mean"].values
    sd  = sub["test_acc_std"].fillna(0).values
    bars = ax.bar(x + (i - 1) * w, mu, w, color=style["color"],
                  label=style["label"], alpha=0.85, edgecolor="white")
    ax.errorbar(x + (i - 1) * w, mu, yerr=sd, fmt="none",
                color="black", capsize=4, linewidth=1.2)
    for bar, m in zip(bars, mu):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.015,
                f"{m:.2f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x); ax.set_xticklabels(REGIME_LABELS)
ax.set_ylim(0, 1.1); ax.set_ylabel("Test Accuracy")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eval_bars_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eval_bars_accuracy.png")


In [ ]:
# ── Delta: LFR gain over supervised baseline ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Accuracy gain over Supervised baseline (LFR − Supervised)", fontsize=13, fontweight="bold")

for ax, (method, title) in zip(axes, [
    ("lfr_linear_eval",   "LFR Linear Eval − Supervised"),
    ("lfr_full_finetune", "LFR Full Finetune − Supervised"),
]):
    sup  = stats[stats["method"] == "supervised"].set_index("regime").loc[REGIME_ORDER]["test_acc_mean"].values
    lfr  = stats[stats["method"] == method].set_index("regime").loc[REGIME_ORDER]["test_acc_mean"].values
    delta = lfr - sup
    colors = ["#2ca02c" if d >= 0 else "#d62728" for d in delta]
    bars = ax.bar(REGIME_LABELS, delta, color=colors)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(title); ax.set_ylabel("Δ Accuracy"); ax.grid(axis="y", alpha=0.3)
    for bar, d in zip(bars, delta):
        ax.text(bar.get_x() + bar.get_width()/2,
                d + (0.003 if d >= 0 else -0.008),
                f"{d:+.3f}", ha="center", va="bottom" if d >= 0 else "top", fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eval_delta_vs_supervised.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eval_delta_vs_supervised.png")


In [ ]:
# ── Summary table: mean (std) for all metrics ─────────────────────────────
rows = []
for metric, label in [("test_acc","Acc"), ("test_f1","F1"), ("test_recall","Recall"), ("test_precision","Precision")]:
    for method in METHODS:
        sub = stats[stats["method"] == method].set_index("regime").loc[REGIME_ORDER]
        for regime in REGIME_ORDER:
            mu = sub.loc[regime, f"{metric}_mean"]
            sd = sub.loc[regime, f"{metric}_std"] if not pd.isna(sub.loc[regime, f"{metric}_std"]) else 0
            rows.append({"Metric": label, "Method": method, "Regime": regime,
                         "Mean": round(mu, 4), "Std": round(sd, 4),
                         "Summary": f"{mu:.3f} ± {sd:.3f}"})

summary_df = pd.DataFrame(rows)
pivot_summary = summary_df.pivot_table(
    index=["Metric","Regime"], columns="Method", values="Summary", aggfunc="first"
)
print(pivot_summary.to_string())
